# YogaPose StyleGAN2-ADA Colab (A100-ready)

This notebook trains a **StyleGAN2-ADA** model on a **subset of 5–10 yoga pose classes** and then generates + displays synthetic samples.

## Key reliability fixes in this version
- Verifies the Kaggle zip is a real ZIP (not an HTML login page)
- Prints dataset discovery + class folder names
- Writes **all outputs** under a single Drive root
- Adds explicit checks for: training zip contents, run dirs, snapshots, generated images
- Makes the `generate.py` call use quoted paths to avoid issues with spaces

If generation shows **0 images**, the new diagnostic cells will tell you whether: training never produced snapshots, the wrong snapshot is selected, or output is going to a different folder.

---
## Cell 1 — Mount Drive and configure paths

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=False)

# Root folder for this notebook's outputs
DRIVE_ROOT = '/content/drive/MyDrive/YogaPoseStyleGAN'

# Dataset zip location (you can change this to wherever you stored it)
DATA_ZIP = os.path.join('/content/drive/MyDrive/computer-vision', 'yoga-poses-dataset-107.zip')

EXTRACT_DIR = os.path.join(DRIVE_ROOT, 'dataset')
SUBSET_DIR = os.path.join(DRIVE_ROOT, 'subset_dataset')
STYLEGAN_REPO = '/content/stylegan2-ada-pytorch'
TRAIN_ZIP = os.path.join(DRIVE_ROOT, 'yoga_subset_stylegan.zip')
RUNS_DIR = os.path.join(DRIVE_ROOT, 'training-runs')
GENERATED_DIR = os.path.join(DRIVE_ROOT, 'generated_samples')

for d in [DRIVE_ROOT, EXTRACT_DIR, SUBSET_DIR, RUNS_DIR, GENERATED_DIR]:
    os.makedirs(d, exist_ok=True)

print('Drive mounted ✓')
print('DRIVE_ROOT   =', DRIVE_ROOT)
print('DATA_ZIP     =', DATA_ZIP)
print('EXTRACT_DIR  =', EXTRACT_DIR)
print('SUBSET_DIR   =', SUBSET_DIR)
print('TRAIN_ZIP    =', TRAIN_ZIP)
print('RUNS_DIR     =', RUNS_DIR)
print('GENERATED_DIR=', GENERATED_DIR)

---
## Cell 2 — Install dependencies

In [ ]:
!pip install -q pillow tqdm matplotlib

---
## Cell 3 — Download dataset zip (only if missing) + validate

Kaggle sometimes returns an HTML login page when unauthenticated. This cell checks that the downloaded file is a real ZIP before continuing.

In [ ]:
import zipfile

if not os.path.exists(DATA_ZIP):
    print('Downloading dataset zip...')
    !curl -L https://www.kaggle.com/api/v1/datasets/download/arrowe/yoga-poses-dataset-107 -o "{DATA_ZIP}"
else:
    print('Dataset zip already exists ✓')

# Validate zip
print('Zip size (bytes):', os.path.getsize(DATA_ZIP) if os.path.exists(DATA_ZIP) else None)
try:
    with zipfile.ZipFile(DATA_ZIP, 'r') as zf:
        bad = zf.testzip()
        names = zf.namelist()[:10]
    print('Zip validation ✓')
    print('Example zip entries:', names)
    if bad is not None:
        print('WARNING: first bad file in zip:', bad)
except zipfile.BadZipFile:
    raise RuntimeError('DATA_ZIP is not a valid zip. Kaggle likely returned an HTML login page. Download the zip manually to Drive or switch to kaggle API token download.')

---
## Cell 4 — Extract dataset and discover train directory

We search for a `train/` folder. If your dataset layout differs, this cell prints the extracted top-level folders so you can adjust.

In [ ]:
import glob

train_dirs = glob.glob(os.path.join(EXTRACT_DIR, '**/train'), recursive=True)
if not train_dirs:
    print('Extracting dataset...')
    with zipfile.ZipFile(DATA_ZIP, 'r') as zf:
        zf.extractall(EXTRACT_DIR)
else:
    print('Dataset already extracted ✓')

# Inspect extract dir
print('Top-level items in EXTRACT_DIR:', sorted(os.listdir(EXTRACT_DIR))[:30])

train_dirs = glob.glob(os.path.join(EXTRACT_DIR, '**/train'), recursive=True)
DATASET_DIR = train_dirs[0] if train_dirs else EXTRACT_DIR
print('DATASET_DIR =', DATASET_DIR)

classes = sorted([d for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d))])
print('Total class folders found:', len(classes))
print('First 30 class folder names:', classes[:30])

---
## Cell 5 — Choose 5–10 pose classes + image size

In [ ]:
# ⚙️ IMPORTANT: Update these to match folder names printed in Cell 4
SELECTED_CLASSES = []

# Default suggestion if you haven't chosen yet: pick first 8 classes
if not SELECTED_CLASSES:
    SELECTED_CLASSES = classes[:8]

IMG_SIZE = 256
MAX_IMAGES_PER_CLASS = 300

print('Selected classes:', SELECTED_CLASSES)
print('IMG_SIZE:', IMG_SIZE)
print('MAX_IMAGES_PER_CLASS:', MAX_IMAGES_PER_CLASS)

---
## Cell 6 — Build subset dataset (resize + pad)

In [ ]:
import shutil
from PIL import Image, ImageOps
from tqdm import tqdm

VALID_EXTS = {'.jpg', '.jpeg', '.png', '.jpe'}

# Clear previous subset
if os.path.exists(SUBSET_DIR):
    shutil.rmtree(SUBSET_DIR)
os.makedirs(SUBSET_DIR, exist_ok=True)

def resize_and_pad(img, size):
    img = ImageOps.exif_transpose(img).convert('RGB')
    img.thumbnail((size, size), Image.Resampling.LANCZOS)
    canvas = Image.new('RGB', (size, size), (255, 255, 255))
    x = (size - img.width) // 2
    y = (size - img.height) // 2
    canvas.paste(img, (x, y))
    return canvas

summary = {}
for cls in SELECTED_CLASSES:
    src_dir = os.path.join(DATASET_DIR, cls)
    dst_dir = os.path.join(SUBSET_DIR, cls)
    os.makedirs(dst_dir, exist_ok=True)

    if not os.path.isdir(src_dir):
        print(f'WARNING: class not found: {cls}')
        summary[cls] = 0
        continue

    files = [f for f in sorted(os.listdir(src_dir)) if os.path.splitext(f)[1].lower() in VALID_EXTS]
    files = files[:MAX_IMAGES_PER_CLASS]

    count = 0
    for fname in tqdm(files, desc=cls):
        src = os.path.join(src_dir, fname)
        dst = os.path.join(dst_dir, os.path.splitext(fname)[0] + '.png')
        try:
            img = Image.open(src)
            out = resize_and_pad(img, IMG_SIZE)
            out.save(dst, format='PNG')
            count += 1
        except Exception:
            pass

    summary[cls] = count

total = sum(summary.values())
print('Subset build complete ✓')
print('Total images in subset:', total)
for k, v in summary.items():
    print(f'  {k:30s} {v}')

if total == 0:
    raise RuntimeError('Subset contains 0 images. Fix SELECTED_CLASSES to match actual folder names in DATASET_DIR.')

---
## Cell 7 — Create training ZIP and verify contents

In [ ]:
import zipfile

if os.path.exists(TRAIN_ZIP):
    os.remove(TRAIN_ZIP)

with zipfile.ZipFile(TRAIN_ZIP, 'w', compression=zipfile.ZIP_STORED) as zf:
    for root, _, files in os.walk(SUBSET_DIR):
        for f in files:
            full = os.path.join(root, f)
            rel = os.path.relpath(full, SUBSET_DIR)
            zf.write(full, rel)

with zipfile.ZipFile(TRAIN_ZIP, 'r') as zf:
    names = zf.namelist()

print('Training zip created ✓')
print('Path:', TRAIN_ZIP)
print('Files in zip:', len(names))
print('First 10:', names[:10])

if len(names) == 0:
    raise RuntimeError('Training zip is empty. Subset dataset did not contain any images.')

---
## Cell 8 — Clone StyleGAN2-ADA and install requirements

In [ ]:
if not os.path.exists(STYLEGAN_REPO):
    !git clone https://github.com/NVlabs/stylegan2-ada-pytorch.git "{STYLEGAN_REPO}"
else:
    print('StyleGAN2-ADA repo already cloned ✓')

%cd "{STYLEGAN_REPO}"
!pip install -q -r requirements.txt

---
## Cell 9 — Training settings

In [ ]:
!nvidia-smi

KIMG = 200
SNAP = 20
BATCH = 16
AUG = 'ada'

print('KIMG=', KIMG, 'SNAP=', SNAP, 'BATCH=', BATCH, 'AUG=', AUG)

---
## Cell 10 — Launch training (Drive-backed outdir)

In [ ]:
!python train.py \
  --outdir="{RUNS_DIR}" \
  --data="{TRAIN_ZIP}" \
  --gpus=1 \
  --cfg=auto \
  --snap={SNAP} \
  --kimg={KIMG} \
  --batch={BATCH} \
  --aug={AUG}

---
## Cell 11 — Find snapshot + diagnose runs

In [ ]:
import glob

run_dirs = sorted(glob.glob(os.path.join(RUNS_DIR, '*')))
print('Run dirs found:', len(run_dirs))
print('Last 3 run dirs:', run_dirs[-3:])

latest_run = run_dirs[-1] if run_dirs else None
print('Latest run:', latest_run)

snapshots = sorted(glob.glob(os.path.join(latest_run, 'network-snapshot-*.pkl'))) if latest_run else []
print('Snapshots found:', len(snapshots))
NETWORK_PKL = snapshots[-1] if snapshots else None
print('Latest snapshot:', NETWORK_PKL)

if NETWORK_PKL is None:
    raise RuntimeError('No network snapshot found. Training may have failed or SNAP/KIMG settings prevented snapshots. Inspect the output of Cell 10.')

---
## Cell 12 — Generate images (and verify output)

In [ ]:
# Clear old outputs
for f in glob.glob(os.path.join(GENERATED_DIR, '*.png')):
    os.remove(f)

!python generate.py --outdir="{GENERATED_DIR}" --trunc=0.8 --seeds=0-24 --network="{NETWORK_PKL}"

generated = sorted(glob.glob(os.path.join(GENERATED_DIR, '*.png')))
print('Generated:', len(generated), 'png files')
print('Output dir:', GENERATED_DIR)
if len(generated) == 0:
    raise RuntimeError('generate.py produced 0 images. Check generate.py output above for errors and confirm NETWORK_PKL is valid.')

---
## Cell 13 — Display generated images inline

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

sample_files = sorted(glob.glob(os.path.join(GENERATED_DIR, '*.png')))
print('Found', len(sample_files), 'generated images in', GENERATED_DIR)

n = min(25, len(sample_files))
grid = 5

fig, axes = plt.subplots(grid, grid, figsize=(12, 12))
axes = axes.flatten()

for i in range(grid*grid):
    ax = axes[i]
    if i < n:
        f = sample_files[i]
        ax.imshow(mpimg.imread(f))
        ax.set_title(os.path.basename(f), fontsize=7)
    ax.axis('off')

plt.suptitle('Generated YogaPose Samples (StyleGAN2-ADA)', fontsize=14)
plt.tight_layout()
plt.show()